# Bible Scraping Project
Extraction très propre du texte de la Bible (seulement les chapitres/versets, sans en-tête ni annotations).

In [1]:
import requests
from bs4 import BeautifulSoup
import re
import time
import urllib3
from IPython.display import clear_output

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36"
}

In [2]:
def get_bible_books():
    url = "https://nybaiboly.net/Bible.htm"
    r = requests.get(url, headers=HEADERS, verify=False)
    r.encoding = 'utf-8'
    soup = BeautifulSoup(r.text, 'html.parser')
    links = soup.find_all('a')
    book_links = [a['href'] for a in links if a.has_attr('href') and 'BibleMalagasyHtm' in a['href']]
    # Remove duplicates
    seen = set()
    unique_links = []
    for link in book_links:
        if link not in seen:
            seen.add(link)
            unique_links.append(link)
    return unique_links

def scrape_book(book_url_suffix):
    url = f"https://nybaiboly.net/{book_url_suffix}"
    try:
        r = requests.get(url, headers=HEADERS, verify=False, timeout=15)
        r.encoding = 'utf-8'
        soup = BeautifulSoup(r.text, 'html.parser')
        
        # Seulement les paragraphes et divs contenant le texte Biblique ('Usuel' ou 'Posie')
        paragraphs = soup.find_all(['p', 'div'])
        cleaned_verses = []
        
        allowed_classes = {'Usuel', 'Posie'}
        
        for p in paragraphs:
            c = p.get('class')
            if c:
                # Si l'élément a la classe Usuel ou Posie
                if any(cls in allowed_classes for cls in c):
                    text = p.get_text(separator=' ', strip=True)
                    if not text: continue
                    
                    # 1. Retirer les crochets et leur contenu (ex: [ny namoronan'andriamanitra...])
                    text = re.sub(r'\[.*?\]\s*', '', text)
                    
                    # 2. Retirer les numéros de versets au début (ex: "1 ")
                    text = re.sub(r'^\d+\s*', '', text)
                    
                    # 3. Mettre tout en minuscule
                    text = text.lower()
                    
                    # 4. Garder seulement les textes qui ont du sens (pas juste un caractère)
                    if len(text) > 3:
                        cleaned_verses.append(text)
                        
        return cleaned_verses
    except Exception as e:
        print(f"Erreur pour {book_url_suffix}: {e}")
        return []


In [3]:
book_links = get_bible_books()
print(f"Le site contient {len(book_links)} livres à scraper.")

all_bible_text = []

for i, link in enumerate(book_links):
    verses = scrape_book(link)
    all_bible_text.extend(verses)
    time.sleep(0.5)
    
    if (i + 1) % 5 == 0 or i == len(book_links) - 1:
        clear_output(wait=True)
        print(f"Progression : {i+1}/{len(book_links)} livres traités...")

# Écriture dans un fichier texte brut
output_file = 'bible_malagasy_brute.txt'
with open(output_file, 'w', encoding='utf-8') as f:
    for verse in all_bible_text:
        f.write(verse + "\n")
        
print(f"\n✅ Scraping super propre terminé. {len(all_bible_text)} lignes ont été sauvegardées dans '{output_file}'.")

Progression : 66/66 livres traités...

✅ Scraping super propre terminé. 31061 lignes ont été sauvegardées dans 'bible_malagasy_brute.txt'.
